# FEniCS Base Setup for PI-DeepONet Ablation Study (Kaggle Notebook)
This notebook generates high-fidelity Finite Element (FEM) baseline solutions for two benchmark 2D linear elasticity geometries:
1. **Geometry 1**: Square plate ($2\text{m} \times 2\text{m}$) with circular hole ($R=0.25\text{m}$) under uniaxial tension.
2. **Geometry 2**: Rectangular pressure vessel ($3\text{m} \times 1.5\text{m}$) with nozzle cutout ($R=0.2\text{m}$) under biaxial tension.

Outputs are evaluated on a $50 \times 50$ Cartesian grid and saved as structured CSV files in `/kaggle/working/`.

In [ ]:
# ── CELL 1: KAGGLE ENVIRONMENT INSTALL BLOCK ─────────────────────────────────
# Kaggle runs Ubuntu Linux with pip and apt-get permissions.
# Legacy FEniCS (dolfin + mshr) installs cleanly via APT packages.
import os
import sys

print("Installing legacy FEniCS (dolfin + mshr) via APT package manager...")
os.system("apt-get update -qq && apt-get install -y -qq fenics python3-fenics-dolfin python3-mshr > /dev/null 2>&1")

# Append standard Ubuntu Python site-packages directory to sys.path
ubuntu_py_path = "/usr/lib/python3/dist-packages"
if os.path.exists(ubuntu_py_path) and ubuntu_py_path not in sys.path:
    sys.path.append(ubuntu_py_path)

# Verify installation
try:
    import dolfin
    import mshr
    print(f"✅ FEniCS DOLFIN version: {dolfin.__version__} loaded successfully!")
except ImportError:
    print("⚠️ Direct import failed. Running fallback pip installation...")
    os.system("pip install fenics -q")
    import dolfin
    import mshr
    print(f"✅ FEniCS DOLFIN version: {dolfin.__version__} loaded!")

In [ ]:
# ── CELL 2: IMPORTS & GLOBAL PARAMETERS ───────────────────────────────────────
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dolfin
import mshr

# Set output directory to Kaggle working directory
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset target export path: {OUTPUT_DIR}")

In [ ]:
# ── CELL 3: MATERIAL & MESH SETUP FUNCTIONS ───────────────────────────────────
# Material Properties: Steel (E = 200 GPa = 200,000 MPa, nu = 0.3)
E_MODULUS = 200000.0     # MPa
POISSON_RATIO = 0.3

# Plane Stress Lame Parameters
LMBDA = (E_MODULUS * POISSON_RATIO) / (1.0 - POISSON_RATIO**2)
MU = E_MODULUS / (2.0 * (1.0 + POISSON_RATIO))

def epsilon(u):
    """Symmetric strain tensor."""
    return dolfin.sym(dolfin.grad(u))

def sigma(u):
    """Stress tensor under 2D Plane Stress Hooke's Law."""
    return LMBDA * dolfin.tr(epsilon(u)) * dolfin.Identity(2) + 2.0 * MU * epsilon(u)

def von_mises_stress(s_xx, s_yy, s_xy):
    """Calculate 2D plane stress von Mises stress."""
    return np.sqrt(s_xx**2 - s_xx*s_yy + s_yy**2 + 3.0*s_xy**2)

def create_geometry_1_mesh(width=2.0, height=2.0, radius=0.25, base_res=35):
    """
    Generate mesh for Geometry 1 (Plate with circular hole) with 3x local refinement.
    """
    domain = mshr.Rectangle(dolfin.Point(-width/2.0, -height/2.0), dolfin.Point(width/2.0, height/2.0)) - \
             mshr.Circle(dolfin.Point(0.0, 0.0), radius)
    mesh = mshr.generate_mesh(domain, base_res)
    
    # 3x local refinement near the hole (within 2x radius)
    for _ in range(2):
        cell_markers = dolfin.MeshFunction("bool", mesh, mesh.topology().dim())
        cell_markers.set_all(False)
        for cell in dolfin.cells(mesh):
            p = cell.midpoint()
            r = np.sqrt(p.x()**2 + p.y()**2)
            if r < 2.0 * radius:
                cell_markers[cell] = True
        mesh = dolfin.refine(mesh, cell_markers)
    return mesh

def create_geometry_2_mesh(width=3.0, height=1.5, radius=0.2, base_res=35):
    """
    Generate mesh for Geometry 2 (Pressure vessel nozzle cutout) with 3x local refinement.
    """
    domain = mshr.Rectangle(dolfin.Point(-width/2.0, -height/2.0), dolfin.Point(width/2.0, height/2.0)) - \
             mshr.Circle(dolfin.Point(0.0, 0.0), radius)
    mesh = mshr.generate_mesh(domain, base_res)
    
    # 3x local refinement near cutout
    for _ in range(2):
        cell_markers = dolfin.MeshFunction("bool", mesh, mesh.topology().dim())
        cell_markers.set_all(False)
        for cell in dolfin.cells(mesh):
            p = cell.midpoint()
            r = np.sqrt(p.x()**2 + p.y()**2)
            if r < 2.5 * radius:
                cell_markers[cell] = True
        mesh = dolfin.refine(mesh, cell_markers)
    return mesh

In [ ]:
# ── CELL 4: GEOMETRY 1 SOLVER LOOP (PLATE WITH HOLE) ────────────────────────
# Solves 5 load cases under uniaxial tension sigma_0 = [10, 25, 50, 75, 100] MPa.
# Exports fields on 50x50 grid and computes Kirsch analytical validation error.

load_cases_geo1 = [10.0, 25.0, 50.0, 75.0, 100.0]
mesh_geo1 = create_geometry_1_mesh(width=2.0, height=2.0, radius=0.25)
print(f"Geometry 1 Mesh generated: {mesh_geo1.num_cells()} cells.")

geo1_dfs = {}

for sigma_0 in load_cases_geo1:
    V = dolfin.VectorFunctionSpace(mesh_geo1, "P", 2)
    u = dolfin.TrialFunction(V)
    v = dolfin.TestFunction(V)
    
    # Boundary Subdomains
    class RightBoundary(dolfin.SubDomain):
        def inside(self, x, on_b): return on_b and dolfin.near(x[0], 1.0)
    class LeftBoundary(dolfin.SubDomain):
        def inside(self, x, on_b): return on_b and dolfin.near(x[0], -1.0)
    class LeftCenterPoint(dolfin.SubDomain):
        def inside(self, x, on_b): return dolfin.near(x[0], -1.0) and dolfin.near(x[1], 0.0, eps=0.05)
        
    boundaries = dolfin.MeshFunction("size_t", mesh_geo1, mesh_geo1.topology().dim() - 1)
    boundaries.set_all(0)
    RightBoundary().mark(boundaries, 1)
    ds = dolfin.Measure("ds", domain=mesh_geo1, subdomain_data=boundaries)
    
    bc_left = dolfin.DirichletBC(V.sub(0), dolfin.Constant(0.0), LeftBoundary())
    bc_pin = dolfin.DirichletBC(V.sub(1), dolfin.Constant(0.0), LeftCenterPoint(), method="pointwise")
    bcs = [bc_left, bc_pin]
    
    a = dolfin.inner(sigma(u), epsilon(v)) * dolfin.dx
    L = dolfin.dot(dolfin.Constant((sigma_0, 0.0)), v) * ds(1)
    
    u_sol = dolfin.Function(V)
    dolfin.solve(a == L, u_sol, bcs)
    
    # Project Stress for 50x50 Grid Evaluation
    W = dolfin.TensorFunctionSpace(mesh_geo1, "DG", 1)
    sigma_proj = dolfin.project(sigma(u_sol), W)
    
    xs = np.linspace(-1.0, 1.0, 50)
    ys = np.linspace(-1.0, 1.0, 50)
    records = []
    
    fem_stresses_xx, kirsch_stresses_xx = [], []
    
    for x in xs:
        for y in ys:
            r = np.sqrt(x**2 + y**2)
            if r <= 0.25:
                records.append([x, y, np.nan, np.nan, np.nan, np.nan, np.nan, sigma_0])
            else:
                try:
                    pt = dolfin.Point(x, y)
                    u_val = u_sol(pt)
                    s_mat = sigma_proj(pt)
                    s_xx, s_yy, s_xy = s_mat[0], s_mat[3], s_mat[1]
                    records.append([x, y, s_xx, s_yy, s_xy, u_val[0], u_val[1], sigma_0])
                    
                    # Kirsch Analytical calculation
                    theta = np.arctan2(y, x)
                    R = 0.25
                    s_rr = (sigma_0 / 2.0) * (1.0 - (R / r)**2) + (sigma_0 / 2.0) * (1.0 - 4.0 * (R / r)**2 + 3.0 * (R / r)**4) * np.cos(2.0 * theta)
                    s_tt = (sigma_0 / 2.0) * (1.0 + (R / r)**2) - (sigma_0 / 2.0) * (1.0 + 3.0 * (R / r)**4) * np.cos(2.0 * theta)
                    tau_rt = -(sigma_0 / 2.0) * (1.0 + 2.0 * (R / r)**2 - 3.0 * (R / r)**4) * np.sin(2.0 * theta)
                    s_xx_k = s_rr * np.cos(theta)**2 + s_tt * np.sin(theta)**2 - 2.0 * tau_rt * np.sin(theta) * np.cos(theta)
                    
                    fem_stresses_xx.append(s_xx)
                    kirsch_stresses_xx.append(s_xx_k)
                except Exception:
                    records.append([x, y, np.nan, np.nan, np.nan, np.nan, np.nan, sigma_0])
                    
    df = pd.DataFrame(records, columns=["x", "y", "sigma_xx", "sigma_yy", "sigma_xy", "u_x", "u_y", "load_value"])
    file_name = f"plate_hole_{int(sigma_0)}MPa.csv"
    file_path = os.path.join(OUTPUT_DIR, file_name)
    df.to_csv(file_path, index=False)
    geo1_dfs[sigma_0] = df
    
    # Validation Checks
    l2_err = np.linalg.norm(np.array(fem_stresses_xx) - np.array(kirsch_stresses_xx)) / np.linalg.norm(np.array(kirsch_stresses_xx))
    valid_stresses = df.dropna(subset=["sigma_xx"])
    max_vm = von_mises_stress(valid_stresses["sigma_xx"], valid_stresses["sigma_yy"], valid_stresses["sigma_xy"]).max()
    
    print(f"➡️ Load {int(sigma_0)} MPa | Saved: {file_name} | Kirsch L2 Error: {l2_err:.4e} | Max von Mises: {max_vm:.2f} MPa")

In [ ]:
# ── CELL 5: GEOMETRY 2 SOLVER LOOP (PRESSURE VESSEL NOZZLE CUTOUT) ──────────
# Solves 5 pressure cases under biaxial tension p = [1, 2, 5, 10, 20] MPa.
# Exports fields on 50x50 grid and computes maximum von Mises stress per run.

pressure_cases_geo2 = [1.0, 2.0, 5.0, 10.0, 20.0]
mesh_geo2 = create_geometry_2_mesh(width=3.0, height=1.5, radius=0.2)
print(f"Geometry 2 Mesh generated: {mesh_geo2.num_cells()} cells.")

geo2_dfs = {}

for p_val in pressure_cases_geo2:
    V = dolfin.VectorFunctionSpace(mesh_geo2, "P", 2)
    u = dolfin.TrialFunction(V)
    v = dolfin.TestFunction(V)
    
    class LeftEdge(dolfin.SubDomain):
        def inside(self, x, on_b): return on_b and dolfin.near(x[0], -1.5)
    class RightEdge(dolfin.SubDomain):
        def inside(self, x, on_b): return on_b and dolfin.near(x[0], 1.5)
    class BottomEdge(dolfin.SubDomain):
        def inside(self, x, on_b): return on_b and dolfin.near(x[1], -0.75)
    class TopEdge(dolfin.SubDomain):
        def inside(self, x, on_b): return on_b and dolfin.near(x[1], 0.75)
        
    boundaries = dolfin.MeshFunction("size_t", mesh_geo2, mesh_geo2.topology().dim() - 1)
    boundaries.set_all(0)
    LeftEdge().mark(boundaries, 1)
    RightEdge().mark(boundaries, 2)
    BottomEdge().mark(boundaries, 3)
    TopEdge().mark(boundaries, 4)
    
    ds = dolfin.Measure("ds", domain=mesh_geo2, subdomain_data=boundaries)
    
    # Pin symmetry points to prevent rigid body translation
    class XPin(dolfin.SubDomain):
        def inside(self, x, on_b): return dolfin.near(x[0], -1.5) and dolfin.near(x[1], 0.0, eps=0.05)
    class YPin(dolfin.SubDomain):
        def inside(self, x, on_b): return dolfin.near(x[0], 0.0, eps=0.05) and dolfin.near(x[1], -0.75)
        
    bc_x = dolfin.DirichletBC(V.sub(0), dolfin.Constant(0.0), XPin(), method="pointwise")
    bc_y = dolfin.DirichletBC(V.sub(1), dolfin.Constant(0.0), YPin(), method="pointwise")
    bcs = [bc_x, bc_y]
    
    a = dolfin.inner(sigma(u), epsilon(v)) * dolfin.dx
    L = dolfin.dot(dolfin.Constant((-p_val, 0.0)), v) * ds(1) + \
        dolfin.dot(dolfin.Constant((p_val, 0.0)), v) * ds(2) + \
        dolfin.dot(dolfin.Constant((0.0, -p_val)), v) * ds(3) + \
        dolfin.dot(dolfin.Constant((0.0, p_val)), v) * ds(4)
        
    u_sol = dolfin.Function(V)
    dolfin.solve(a == L, u_sol, bcs)
    
    # Grid evaluation (50x50 over [-1.5, 1.5] x [-0.75, 0.75])
    W = dolfin.TensorFunctionSpace(mesh_geo2, "DG", 1)
    sigma_proj = dolfin.project(sigma(u_sol), W)
    
    xs = np.linspace(-1.5, 1.5, 50)
    ys = np.linspace(-0.75, 0.75, 50)
    records = []
    
    for x in xs:
        for y in ys:
            r = np.sqrt(x**2 + y**2)
            if r <= 0.2:
                records.append([x, y, np.nan, np.nan, np.nan, np.nan, np.nan, p_val])
            else:
                try:
                    pt = dolfin.Point(x, y)
                    u_val = u_sol(pt)
                    s_mat = sigma_proj(pt)
                    records.append([x, y, s_mat[0], s_mat[3], s_mat[1], u_val[0], u_val[1], p_val])
                except Exception:
                    records.append([x, y, np.nan, np.nan, np.nan, np.nan, np.nan, p_val])
                    
    df = pd.DataFrame(records, columns=["x", "y", "sigma_xx", "sigma_yy", "sigma_xy", "u_x", "u_y", "load_value"])
    file_name = f"vessel_{int(p_val)}MPa.csv"
    file_path = os.path.join(OUTPUT_DIR, file_name)
    df.to_csv(file_path, index=False)
    geo2_dfs[p_val] = df
    
    valid_stresses = df.dropna(subset=["sigma_xx"])
    max_vm = von_mises_stress(valid_stresses["sigma_xx"], valid_stresses["sigma_yy"], valid_stresses["sigma_xy"]).max()
    
    print(f"➡️ Pressure {int(p_val)} MPa | Saved: {file_name} | Max von Mises: {max_vm:.2f} MPa")

In [ ]:
# ── CELL 6: CSV EXPORT SUMMARY & VALIDATION PRINTS ──────────────────────────
# Inspects exported CSV datasets, verifies schema integrity, and plots stress fields.

print("\n=== DATASET GENERATION SUMMARY ===")
exported_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".csv")])
print(f"Total CSV datasets exported to {OUTPUT_DIR}: {len(exported_files)}")
for fname in exported_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    df_temp = pd.read_csv(fpath)
    valid_cnt = df_temp["sigma_xx"].notnull().sum()
    print(f"  • {fname:<24} | Rows: {len(df_temp)} | Valid Grid Nodes: {valid_cnt}")

# Plot representative sample contour (Geometry 1, 100 MPa)
plt.figure(figsize=(10, 4.5))
df_sample = geo1_dfs[100.0]

plt.subplot(1, 2, 1)
sc1 = plt.scatter(df_sample['x'], df_sample['y'], c=df_sample['sigma_xx'], cmap='jet', s=15)
plt.colorbar(sc1, label=r'$\sigma_{xx}$ (MPa)')
plt.title(r'Geometry 1: $\sigma_{xx}$ under 100 MPa Uniaxial Tension')
plt.axis('equal')

plt.subplot(1, 2, 2)
df_sample2 = geo2_dfs[20.0]
vm_stress = von_mises_stress(df_sample2['sigma_xx'], df_sample2['sigma_yy'], df_sample2['sigma_xy'])
sc2 = plt.scatter(df_sample2['x'], df_sample2['y'], c=vm_stress, cmap='magma', s=15)
plt.colorbar(sc2, label=r'von Mises Stress (MPa)')
plt.title(r'Geometry 2: von Mises Stress under 20 MPa Biaxial Pressure')
plt.axis('equal')

plt.tight_layout()
plt.show()
print("\n✅ Dataset generation complete and verified!")